In [2]:
import polars as pl
import pandas as pd
import glob
import re
import os
import yaml
import pysam

In [1]:
# config_path = "/user/leuven/350/vsc35059/osiga/ASAP/asap_git/wasp_pipeline/test/config.yaml"
# config = yaml.safe_load(open(config_path))
# analysis_dir = config["main"]["analysis_dir"]
# print(analysis_dir) 
# bcf_path = config["data"]["bcf_combined"]

In [5]:

analysis_dir = "/staging/leuven/stg_00090/ASA/analysis/2024_T2T_ATAC_analysis/20250107_allelic_analysis"

# filtered VCF of all donors, in peaks (SNPs and indels)
bcf_path = f"{analysis_dir}/WGS_chm13_BCFs.missing_to_ref.norm.subset_peaks_exended2500bp.vcf.gz"
bcf = pysam.VariantFile(bcf_path)

# get contig names and samples from the bcf 
chroms = [x for x in bcf.header.contigs]
samples = [x for x in bcf.header.samples]



In [6]:
# makeoutput directory
outdir = analysis_dir + "/phasing_blocks/"
os.makedirs(f"{outdir}/per_chrom", exist_ok=True)

# parse BCF file and extract phasing information for each sample
for chrom in chroms:

    print(chrom)
    phasing_dict = {}
    i = 0

    # extract phasing info for all samples per chromosome
    for record in bcf.fetch(contig=chrom):

        for sample in record.samples:

            try:
                phasing_block = record.samples[sample]['PS']
                if phasing_block:

                    l = [record.chrom, record.pos, sample, record.samples[sample]['PS']]
                    if sample in phasing_dict:
                        phasing_dict[sample].append(l)
                    else:
                        phasing_dict[sample] = [l]
            except:
                # skip unphased variants
                continue
            
        if i % 200000 == 0:
            print(i)
        i += 1

    # write phasing blocks to csv files for each sample
    for sample in samples:

        # convert list of lists to dataframe
        try:
            df = pd.DataFrame(phasing_dict[sample], columns=['chrom', 'pos', 'sample', 'phasing_block'])
            df = pl.from_pandas(df)

            (df
                .select("chrom", "pos", "phasing_block")
                .group_by(["chrom", "phasing_block"])
                .agg(pl.col("pos").min().alias("start"), 
                    pl.col("pos").max().alias("end"))
                .select(["chrom", "start", "end", "phasing_block"])
                .sort("chrom", "start")
                .write_csv(f"{outdir}/per_chrom/pb_{sample}_{chrom}.bed", separator = "\t", include_header = False)
            )
        except:
            #print(f"Sample {sample} not found in phasing_dict")
            continue

chr1
0
200000
400000
600000
800000
1000000
1200000
1400000
chr2
0
200000
400000
600000
800000
1000000
1200000
1400000
chr3
0
200000
400000
600000
800000
1000000
1200000
chr4
0
200000
400000
600000
800000
1000000
chr5
0
200000
400000
600000
800000
1000000
chr6
0
200000
400000
600000
800000
1000000
chr7
0
200000
400000
600000
800000
1000000
chr8
0
200000
400000
600000
800000
chr9
0
200000
400000
600000
chr10
0
200000
400000
600000
800000
chr11
0
200000
400000
600000
800000
chr12
0
200000
400000
600000
800000
chr13
0
200000
400000
chr14
0
200000
400000
chr15
0
200000
400000
chr16
0
200000
400000
600000
chr17
0
200000
400000
600000
chr18
0
200000
400000
chr19
0
200000
400000
chr20
0
200000
400000
chr21
0
200000
chr22
0
200000
chrX
0
200000
400000
600000
800000
chrY
0
chrM


## Merge all chromosomes per sample

In [18]:
# extract sample names (ASA_[0-9]+) from the phasing block files
sample_names = []
for file in glob.glob(f"{outdir}/per_chrom/*_chr*.bed"):
    sample = re.search(r"pb_(ASA_[0-9]+)_chr*", file).group(1)
    sample_names.append(sample)

phased_samples = set(sample_names)

In [21]:
# concatenate all chromosomes for each sample (in bash)

outdir="/staging/leuven/stg_00090/ASA/analysis/2024_T2T_ATAC_analysis/20250107_allelic_analysis/phasing_blocks"
!mkdir -p {outdir}/merged


for sample in phased_samples:
    # check if files exist
    path = f"{outdir}/pb_{sample}_*.bed"
    !cat {outdir}/per_chrom/pb_{sample}_chr*.bed > {outdir}/merged/pb_{sample}.bed


In [47]:
## Testing for TGFA gene locus
bcf = pysam.VariantFile(bcf_path)

for record in bcf.fetch('chr2', 70481136, 70481822):

    for sample in ["ASA_026", "ASA_040", "ASA_106"]: 
       

        try:
            phasing_block = record.samples[sample]['PS']
            if phasing_block:

                l = [record.chrom, record.pos, record.ref, record.alts[0], sample, record.samples[sample]['GT'], record.samples[sample]['PS']]
                print(sample)
                print(l)
        except:
            # skip unphased variants
            continue

ASA_026
['chr2', 70481258, 'G', 'A', 'ASA_026', (0, 1), 67078480]
ASA_040
['chr2', 70481258, 'G', 'A', 'ASA_040', (0, 1), 69924544]
ASA_106
['chr2', 70481258, 'G', 'A', 'ASA_106', (0, 1), 70258063]
ASA_026
['chr2', 70481311, 'C', 'A', 'ASA_026', (1, 0), 67078480]
ASA_040
['chr2', 70481311, 'C', 'A', 'ASA_040', (1, 0), 69924544]
ASA_106
['chr2', 70481311, 'C', 'A', 'ASA_106', (1, 0), 70258063]
